# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My lane (from Week 1–2):** does a page's CTR under-perform what it should get, given its content
type and how reachable it actually is in search? Two signals feed that question directly, so
those are the two I check before trusting them.

**Signal A — CTR vs. position bucket (flag-linked).** This is the signal behind FlyRank's real
`needs_ctr_fix` flag and behind this repo's own baseline pipeline (`low_ctr_visible_page` in
`scripts/02_baseline_score.py`), which treats any position ≤ 20 as "should be getting more
clicks than it is." The assumption baked into that flag is: **better position → higher CTR,
roughly monotonically.**

**Signal B — CTR vs. content_type.** From my own Week 1/2 question: do certain content formats
structurally over- or under-perform on CTR, independent of position? This isn't a FlyRank
product flag, but it's the signal my whole lane is built on.

**The rule (plain words):** A page is a CTR-fix candidate if (1) it's getting real search
visibility, (2) it's actually ranking somewhere reachable (position 1–20, not buried), and
(3) its measured CTR sits meaningfully below the *typical* CTR for its own content type — not
below some assumed "position X should get Y% CTR" curve, because Signal A below shows that curve
isn't reliable at the very top.

**Reason code (one, always):** `ctr_underperforms_type_peer_at_reachable_position`
**Action labels:** `review_ctr` (score ≥ 0.20) or `monitor` (below threshold)

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows, {df['client_id'].nunique()} clients")

# Same visibility floor the rest of the repo's pipeline uses: only score pages
# that are actually getting found in search.
active = df[df["impressions_90d"] >= 100].copy()
print(f"Active rows (impressions_90d >= 100): {len(active):,}")

Loaded 30,000 rows, 32 clients
Active rows (impressions_90d >= 100): 22,006


### Signal A — CTR by position bucket (bucket table, n printed)

In [2]:
pos_bucket = pd.cut(
    active["avg_position"].replace(0, np.nan),   # 0 means "no position data", not rank 0
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["top_3 (<=3)", "page_1 (4-10)", "striking (11-20)", "page_3_5 (21-50)", "deep (50+)"],
)
signal_a = (
    active.groupby(pos_bucket, observed=True)["ctr"]
    .agg(n="count", mean_ctr="mean", median_ctr="median")
    .round(3)
)
print(signal_a)

corr = active.loc[active["avg_position"] > 0, ["avg_position", "ctr"]].corr().iloc[0, 1]
print(f"\nPearson corr(avg_position, ctr), position>0 rows: {corr:.3f}")

fine = (
    active[(active["avg_position"] > 0) & (active["avg_position"] <= 10)]
    .assign(pos_round=lambda d: d["avg_position"].round(0))
    .groupby("pos_round")["ctr"].agg(n="count", mean_ctr="mean").round(3)
)
print("\nFine-grained check, positions 1-10 (does CTR fall smoothly as position worsens?):")
print(fine)

                     n  mean_ctr  median_ctr
avg_position                                
top_3 (<=3)        555     0.337        0.19
page_1 (4-10)     8660     0.354        0.23
striking (11-20)  5876     0.256        0.15
page_3_5 (21-50)  6037     0.142        0.06
deep (50+)         878     0.055        0.00

Pearson corr(avg_position, ctr), position>0 rows: -0.239

Fine-grained check, positions 1-10 (does CTR fall smoothly as position worsens?):
              n  mean_ctr
pos_round                
0.0           4     0.088
1.0          46     0.122
2.0         291     0.343
3.0         466     0.421
4.0        1030     0.444
5.0        1253     0.431
6.0        1689     0.356
7.0        1359     0.330
8.0        1529     0.293
9.0         957     0.309
10.0        591     0.274


**Verdict: MIXED.**

The broad direction is there — `page_1` (0.354 mean CTR, n=8,660) clearly beats `deep` (0.055,
n=878) — but it is **not monotonic at the top**, and the correlation is weak
(r ≈ -0.24, position>0 rows only). The fine-grained view makes it obvious why: mean CTR
*peaks around position 4–5* (~0.43–0.44), not position 1 (0.12, though n=46 there is thin).
`top_3` as a whole (0.337) even sits slightly *below* `page_1` (0.354).

So "lower position number always means higher CTR" — the assumption the reference pipeline's
`low_ctr_visible_page` flag leans on — is only roughly true, not reliably true rank-by-rank.
**This is why the rule below uses position only as a coarse reachability gate (1–20), and reads
CTR itself directly rather than inferring an "expected" CTR from position.** A clean confirm
would have let me skip checking CTR directly and just trust the position band; the mixed result
is what stopped that shortcut.

### Signal B — CTR by content_type (bucket table, n printed)

In [3]:
signal_b = (
    active.groupby("content_type", observed=True)["ctr"]
    .agg(n="count", mean_ctr="mean", median_ctr="median")
    .round(3)
    .sort_values("mean_ctr")
)
print(signal_b)

                        n  mean_ctr  median_ctr
content_type                                   
comparison article    366     0.134       0.000
keyword article     21288     0.252       0.150
feedly article        352     0.706       0.225


**Verdict: CONFIRMED.**

Content type is a real, large, well-supported split: `comparison article` (0.134 mean CTR,
n=366) → `keyword article` (0.252, n=21,288) → `feedly article` (0.706, n=352). `keyword article`
carries the overwhelming majority of rows, so its median (0.150) is the most trustworthy
"typical CTR" reference point — that's what the rule below compares each page against.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
from pathlib import Path

# Readable on purpose: three multiplicative gates, no fitted weights.
type_median_ctr = active.groupby("content_type")["ctr"].transform("median")

# How far below its OWN content type's typical CTR this page sits (Signal B)
ctr_gap = (type_median_ctr - active["ctr"]).clip(lower=0)
active["ctr_gap_norm"] = (ctr_gap / type_median_ctr.replace(0, np.nan)).fillna(0).clip(0, 1)

# Reachable in search at all? Coarse gate only (Signal A showed position isn't a
# reliable proportional predictor of CTR, so it earns a gate, not a weight).
active["position_ok"] = ((active["avg_position"] > 0) & (active["avg_position"] <= 20)).astype(int)

# Worth anyone's time to fix? (visibility, log-scaled, percentile-ranked)
active["visibility_score"] = active["impressions_90d"].rank(pct=True)

active["score"] = (
    active["visibility_score"] * active["position_ok"] * active["ctr_gap_norm"]
).round(4)

active["reason_code"] = "ctr_underperforms_type_peer_at_reachable_position"
active["action"] = np.where(active["score"] >= 0.20, "review_ctr", "monitor")

queue = active.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print("Action counts:")
print(queue["action"].value_counts())
print()
print("Sanity check — does the flagged group actually have lower measured CTR?")
print(queue.groupby("action")["ctr"].agg(n="count", mean_ctr="mean").round(3))

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_cols = [
    "rank", "content_id", "client_id", "content_type",
    "avg_position", "ctr", "impressions_90d",
    "score", "reason_code", "action",
]
queue[out_cols].to_csv(out_path, index=False)
print(f"\nWrote {len(queue):,} ranked rows to {out_path.resolve()}")

Action counts:
action
monitor       18825
review_ctr     3181
Name: count, dtype: int64

Sanity check — does the flagged group actually have lower measured CTR?
                n  mean_ctr
action                     
monitor     18825     0.294
review_ctr   3181     0.037



Wrote 22,006 ranked rows to C:\Users\kara\Desktop\FlyRank\FlyRank_Intern\work\outputs\baseline_action_score.csv


## 3. Top-10 review

*For each of the top ten: the action, why it's there, and what would make it wrong.*

In [5]:
top10 = queue[out_cols].head(10)
top10

,rank,content_id,client_id,content_type,avg_position,ctr,impressions_90d,score,reason_code,action
0,1,content_c8e9d6ab9013,client_19581e27de,keyword article,9.7,0.00,208678,0.9987,ctr_underperforms_type_peer_at_reachable_position,review_ctr
1,2,content_453722754fea,client_f369cb89fc,keyword article,7.6,0.01,140079,0.9301,ctr_underperforms_type_peer_at_reachable_position,review_ctr
2,3,content_f986bd514b6e,client_7f2253d7e2,keyword article,6.6,0.00,22456,0.9299,ctr_underperforms_type_peer_at_reachable_position,review_ctr
3,4,content_4a6607efcb46,client_6208ef0f77,keyword article,2.2,0.01,128068,0.9292,ctr_underperforms_type_peer_at_reachable_position,review_ctr
4,5,content_39881853ef0c,client_f369cb89fc,keyword article,7.2,0.01,112434,0.9274,ctr_underperforms_type_peer_at_reachable_position,review_ctr
5,6,content_d274ac4158ef,client_4e07408562,keyword article,6.8,0.01,65138,0.9178,ctr_underperforms_type_peer_at_reachable_position,review_ctr
6,7,content_e5f459e737b7,client_f369cb89fc,keyword article,5.9,0.01,56363,0.9147,ctr_underperforms_type_peer_at_reachable_position,review_ctr
7,8,content_339b357d04c7,client_bbb965ab0c,keyword article,3.7,0.01,46879,0.9094,ctr_underperforms_type_peer_at_reachable_position,review_ctr
8,9,content_ae6d1339904d,client_7f2253d7e2,keyword article,19.5,0.00,17622,0.9057,ctr_underperforms_type_peer_at_reachable_position,review_ctr
9,10,content_ca17a024f90c,client_4e07408562,keyword article,9.1,0.01,38815,0.9021,ctr_underperforms_type_peer_at_reachable_position,review_ctr


All ten land in the same shape: very high `impressions_90d`, position solidly inside the top
20 (mostly single digits), CTR at or near **0.00–0.01%** — real visibility that almost nobody
clicks on. One line each:

1. **rank 1** — `review_ctr`. 208,678 impressions at position 9.7, CTR 0.00%. Why: massive
   reach, reachable position, essentially zero clicks — the clearest case in the set.
   Would be wrong if: this page ranks for a very broad/ambiguous query where most impressions
   are irrelevant searchers who were never going to click (title/meta can't fix mismatched
   intent).
2. **rank 2** — `review_ctr`. 140,079 impressions at position 7.6, CTR 0.01%. Why: same pattern,
   slightly lower reach. Would be wrong if: the snippet is already being outcompeted by a
   featured snippet or "People also ask" box eating the clicks before they reach any listing.
3. **rank 3** — `review_ctr`. 22,456 impressions at position 6.6, CTR 0.00%. Why: good position,
   good volume, no clicks. Would be wrong if: this is a very recent republish and GSC's 90-day
   window is mixing in a pre-fix period that no longer reflects the live page.
4. **rank 4** — `review_ctr`. 128,068 impressions at position 2.2, CTR 0.01%. Why: this is the
   sharpest anomaly — position 2 with almost no clicks directly illustrates Signal A's finding
   (top position ≠ guaranteed CTR). Would be wrong if: the query is navigational/brand-name and
   users are clicking a sitelink or a different result entirely, not skipping this one out of
   distaste for the title.
5. **rank 5** — `review_ctr`. 112,434 impressions at position 7.2, CTR 0.01%. Why: consistent
   with the pattern above. Would be wrong if: this page cannibalizes clicks with a sibling page
   from the same client ranking just above/below it for the same query.
6. **rank 6** — `review_ctr`. 65,138 impressions at position 6.8, CTR 0.01%. Why: same reasoning,
   smaller but still large volume. Would be wrong if: the keyword's search intent is informational
   and mostly satisfied by the SERP snippet itself (people read the preview, don't need the click).
7. **rank 7** — `review_ctr`. 56,363 impressions at position 5.9, CTR 0.01%. Why: strong position,
   strong volume, weak CTR — textbook fix candidate. Would be wrong if: this is a duplicate/near-
   duplicate URL and the "real" traffic is landing on a canonical version elsewhere in the data.
8. **rank 8** — `review_ctr`. 46,879 impressions at position 3.7, CTR 0.01%. Why: another top-5
   position with almost no clicks — reinforces that position alone doesn't guarantee CTR.
   Would be wrong if: a competitor's rich result (image pack, video carousel) sits directly above
   it and structurally suppresses the click-through regardless of title quality.
9. **rank 9** — `review_ctr`. 17,622 impressions at position 19.5, CTR 0.00%. Why: lower volume
   than the rest but position is right at the edge of "reachable" (19.5) — worth a second look
   before spending effort here. Would be wrong if: position 19.5 is a 90-day *average* masking a
   page that only recently cracked page 2 and hasn't had time to earn clicks yet.
10. **rank 10** — `review_ctr`. 38,815 impressions at position 9.1, CTR 0.01%. Why: same pattern
    as ranks 1–7. Would be wrong if: `content_type` metadata is mis-tagged for this row, so the
    "typical CTR for its type" comparison it was scored against isn't the right peer group.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest pick: rank 9.** It's the one row in the top 10 with noticeably lower volume
(17,622 vs. 40,000+ for the rest) and a position sitting right at the 20-cutoff edge (19.5) —
exactly the kind of borderline case flagged in the row-9 note above. If I tightened the
`position_ok` gate to ≤15 instead of ≤20, this row would drop out; I kept 20 because it matches
the reference pipeline's own `low_ctr_visible_page` threshold, for comparability, but it's the
weakest justified choice in the rule.

**Rank 4 is the most interesting pick, not a weak one** — position 2.2 with 0.01% CTR is exactly
the anomaly Signal A surfaced (CTR does not simply rise as position approaches 1). It's a strong
argument *for* keeping CTR as a directly-measured input rather than inferring it from position.

**Leakage check.** Every input to the score is knowable from the same 90-day trailing snapshot,
before any "did it decline" judgment exists:
- `impressions_90d`, `avg_position`, `ctr`, `content_type` — all observed signals, not derived
  from `trend_pct` / `trend_direction` (the label-source columns per `docs/data-dictionary.md`).
- No FlyRank product flags used anywhere (`health_score`, `needs_ctr_fix`, `is_quick_win` are
  not in this dataset by design — see `skills/flyrank/flyrank-context/SKILL.md`). This rule
  *reproduces* the CTR-fix idea from first principles; it never reads a pre-computed decision.
- No future window: `ctr`, `avg_position`, and `impressions_90d` are all trailing-90-day
  aggregates as of export time — nothing from `impressions_last_30d` / `_prev_30d` (the trend
  inputs) is used, so no half of a forward/backward split leaks into the score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.